In [ ]:
import pandas as pd
import numpy as np

# 参数设置
L = 1.827e-3          # 电极厚度 cm
tau = 60              # 脉冲时间 s
pi = np.pi
coeff = L**2 / (6 * tau)  # 系数 = 9.272e-9 cm^2/s

# 读取数据
df = pd.read_excel('5_010_5.xlsx', sheet_name='Sheet1', skiprows=1)
df.columns = ['工步序号', '工步模式', '起始电压/V', '终止电压/V']

# 提取充电脉冲和静置工步
charge_rows = df[df['工步模式'] == '倍率充电'].reset_index(drop=True)
relax_rows = df[df['工步模式'] == '静置'].reset_index(drop=True)

results = []
for i in range(len(charge_rows)):
    # 充电工步数据
    v0 = charge_rows.loc[i, '起始电压/V']
    v1 = charge_rows.loc[i, '终止电压/V']
    # 对应的静置工步（同索引）
    v2 = relax_rows.loc[i, '终止电压/V']
    v3 = relax_rows.loc[i+1, '终止电压/V']
    
    # 计算 ΔE_t 和 ΔE_s
    delta_E_t = v1 - v0          # 总极化
    delta_E_s = v3 - v2          # 稳态电压变化（取绝对值，实际 v0 > v2）
    delta_E_t_minus_s = v1 + v2 - v0 - v3   # 极化部分
    
    # 避免除零
    if delta_E_t_minus_s <= 0:
        D_eff = np.nan
    else:
        D_eff = coeff * (delta_E_s / delta_E_t_minus_s) 

    if D_eff > 0:
        results.append({
            '脉冲序号': i+1,
            '起始电压/V': v0,
            '终止电压/V': v1,
            '静置终止电压/V': v2,
            'ΔE_t/V': delta_E_t,
            'ΔE_s/V': delta_E_s,
            'ΔE_t-ΔE_s/V': delta_E_t_minus_s,
            'D_eff (cm^2/s)': D_eff
        })

# 输出结果
result_df = pd.DataFrame(results)
result_df.to_excel('GITT_diffusion_coefficients.xlsx', index=False)
print("计算完成，结果已保存至 'GITT_diffusion_coefficients.xlsx'")

计算完成，结果已保存至 'GITT_diffusion_coefficients.xlsx'
